
# **生物医学图像处理 - Project 1提交模板**

同学们好，

本文件是你的作业提交核心模板。你的任务是**在此文件中实现你的图像处理/预测算法**。

**⚠️ 【数据与评测机制说明】 ⚠️**

1.  **关于训练数据**：助教在**之前**已经发放了带有金标准的完整数据集。无论你是调试传统算法，还是训练机器学习模型，都请使用那个数据集。
2.  **关于当前文件夹的模拟数据**：你现在看到的这个目录下的 `Origin` 和 `GT` 文件夹，仅仅是为了让你**模拟最终的隐藏测试环境**，验证你的代码能否顺利跑通的。
3.  **关于最终评测**：你提交代码后，助教会把你写的代码放入一个**只有隐藏 `Origin` 测试集**的全新环境中运行。
4.  **🚨 警告 🚨**：在最终的评测环境中，**代码是绝对无法访问 `GT` 文件夹的**！因此，在此 `submit.ipynb` 中，**严禁编写任何试图读取 `GT` 文件夹的代码**，否则在助教电脑上运行会直接因为找不到路径而崩溃！

---

### **作业流程及提交指南**

#### **第 1 步：准备你的开发环境**

请确保你的电脑上有如下的相对文件夹结构（用于本地模拟测试）：
```text
你的作业文件夹/ (例如: D:\DIP_Project\SubmitExample)
├── Origin/          (模拟环境：极少量的测试图片)
├── GT/              (模拟环境：对应的真值图片，仅供你肉眼对比，代码里别读它！)
├── submit.ipynb     (【必须】本文件，只负责读取Origin并输出到Seg)
├── train.ipynb      (【可选】如果你用了机器学习，这是你的训练代码)
├── my_model.pkl     (【可选】如果你用了机器学习，这是你训练好的模型文件)
└── Seg/             (【自动生成】代码运行后会生成在这里，保证预测得到的文件名字和Origin里的完全相同)
```
*   **路径规范**：你的代码必须依靠上述相对路径运行，**严禁使用绝对路径** (例如 `C:\Users\...`)。
*   **关于同名输出的保证**：你只需要让函数返回处理后的图像矩阵（Numpy数组）。助教提供的测试脚本（Cell 2）会自动保证输出到 Seg 文件夹里的图片与 Origin 里的图片名字完全相同（例如 Origin 里面是 1.png，Seg 得到的也是 1.png）。你不需要自己写保存文件的代码。

#### **第 2 步：编写你的算法 (在下方 Cell 1 中)**

*   所有的预测和处理逻辑必须写在 **`Cell 1`** 的 `process_image` 函数中。
*   允许根据需要编写多函数（辅助函数）。
*   允许导入合理的第三方库（如 `cv2`, `numpy`, `scipy`, `scikit-image`, `scikit-learn` 等）。

#### **第 3 步：关于“机器学习模型”的特殊规定**

如果你决定使用机器学习方法：
1.  **允许算法**：只允许使用传统的机器学习模型（如 SVM, 随机森林等）。**严禁使用深度学习框架**。
2.  **训练数据**：必须使用**之前发放的数据集**进行训练，严禁引入外部数据集。
3.  **代码分离**：训练过程必须写在独立的 `train.ipynb` (或 `.py`) 中。当前这个 `submit.ipynb` **只负责加载本地模型并进行预测**。

#### **第 4 步：本地模拟运行**

*   点击菜单栏的 **`运行 (Run)` -> `全部运行 (Run All Cells)`**。
*   如果一切顺利，代码会自动读取 `Origin` 中的图片，并将结果存入 `Seg` 文件夹。你可以打开 `Seg` 文件夹，与 `GT` 文件夹中的图片做对比，看看算法效果如何。

#### **第 5 步：提交作业**

根据你是否使用了机器学习模型，选择以下**其中一种**方式提交：
*   **【未使用模型 (传统算法)】**：**只提交你写好的 `submit.ipynb` 这一个文件**。
*   **【使用了机器学习模型】**：将以下文件打包成**一个 `.zip` 压缩包**提交：
    1.  `submit.ipynb`（必须包含模型加载和预测逻辑）
    2.  `你的模型文件` (如 `.pkl` 文件)
    3.  `train.ipynb` 或 `train.py` (包含完整训练过程，供助教审查)

**请勿提交 `Origin`, `GT`, `Seg` 文件夹或任何其他图片数据！**
```


```

In [13]:




# Cell 1: 图像处理与预测算法实现 (学生代码区域)
# --------------------------------------------------

# --- 1. 依赖库导入 ---
# 你可以在这里导入你需要的标准库和第三方库
import cv2
import numpy as np
import os
import joblib  # 推荐使用joblib加载sklearn模型

# --- 2. 加载机器学习模型 (可选) ---
# 如果你使用了机器学习模型，请在此处加载。
# 模型文件须与本 submit.ipynb 放在同一根目录下。
MODEL_FILENAME = 'my_model.pkl'  # TODO: 如果使用了模型，请修改为实际的文件名
model = None

# 尝试加载模型
if os.path.exists(MODEL_FILENAME):
    try:
        model = joblib.load(MODEL_FILENAME)
        print(f"✅ 成功加载机器学习模型: {MODEL_FILENAME}")
    except Exception as e:
        print(f"❌ 模型加载失败: {e}")
else:
    print(f"ℹ️ 未检测到模型文件 '{MODEL_FILENAME}'。使用传统算法的同学请忽略此提示。")

# --- 3. 辅助函数定义区域 ---
# 强烈建议将复杂的逻辑拆分为多个辅助函数，以保持代码整洁
def example_preprocess(image):
    """示例预处理函数"""
    return cv2.GaussianBlur(image, (5, 5), 0)

# 放在辅助函数定义区域

# --- 4. 核心处理接口 ---
# ⚠️ 助教评测时只会调用此函数，请勿修改函数名、参数和返回值类型！
def process_image(image_path):
    """
    处理单张图片的函数。
    
    Args:
        image_path (str): 输入图片的完整相对路径 (例如: 'Origin/image1.png')
        
    Returns:
        numpy.ndarray: 处理后的OpenCV图像（应为单通道二值图或灰度图，尺寸须与原图一致）
    """
    # 1. 读取原始图片 (评测环境保证 image_path 指向的文件一定存在)
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)
    if img is None:
        return None 

    # 2. 图像处理与预测算法实现
    # ===================================================================
    # TODO: 请删除下方示例，并在此处填入你的算法。
    # 你可以完全按照自己的思路来实现算法，以下示例仅供参考,甚至判断语句都可以删除。
    # ⚠️ 严禁在此处尝试读取 GT 文件夹！
    
    if model is not None:
        # 【分支A：如果你使用了机器学习模型】
        # (伪代码演示：提取特征 -> 模型预测 -> 还原为图像)
        # gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # features = gray.reshape(-1, 1) 
        # predictions = model.predict(features)
        # processed_image = (predictions.reshape(gray.shape) * 255).astype(np.uint8)
        pass 
        
    else:
        # 【分支B：如果你使用的是传统非学习算法】
        # 步骤1: 预处理
        img_blur = example_preprocess(img)
        # 步骤2: 灰度化
        gray_img = cv2.cvtColor(img_blur, cv2.COLOR_BGR2GRAY)
        # 步骤3: 核心算法 (例如固定阈值或自动阈值)
        _, processed_image = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
    # ===================================================================

    # 3. 返回最终的图像 Numpy 数组
    return processed_image

print("🟢 `process_image` 及相关依赖已成功定义。")

ℹ️ 未检测到模型文件 'my_model.pkl'。使用传统算法的同学请忽略此提示。
🟢 `process_image` 及相关依赖已成功定义。


In [14]:
# Cell 2: 运行与批量处理接口 (模拟评测接口，请勿修改)
# --------------------------------------------------
# ⚠️ 此单元格用于模拟助教的隐藏测试环境。
# 点击“全部运行”时，它会自动读取当前目录下的 Origin 文件夹，
# 调用你的 process_image 处理图片，并将结果存入自动创建的 Seg 文件夹。

import os
import cv2
from tqdm import tqdm
import time

# 定义相对路径 (严禁修改)
base_path = '.' 
origin_path = os.path.join(base_path, 'Origin')
seg_path = os.path.join(base_path, 'Seg')

print("=== 开始模拟批量评测 ===")
start_time = time.time()

# 1. 创建或清空输出文件夹
if not os.path.exists(seg_path):
    os.makedirs(seg_path)
    print(f"📁 输出文件夹 '{seg_path}' 已就绪。")

# 2. 扫描原始图片
valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']
try:
    image_files = [f for f in os.listdir(origin_path) if os.path.splitext(f)[1].lower() in valid_extensions]
    if not image_files:
         print(f"⚠️ 警告：在 '{origin_path}' 中未找到有效图片。请检查模拟文件是否放入该目录。")
    else:
         print(f"🔍 在 '{origin_path}' 中找到 {len(image_files)} 个待处理文件。")
except FileNotFoundError:
    print(f"❌ 致命错误：找不到原始图片文件夹 '{origin_path}'！请确保文件目录结构正确。")
    image_files = [] 

# 3. 循环调用学生算法
if image_files:
    try:
        for filename in tqdm(image_files, desc="图像处理进度"):
            input_image_path = os.path.join(origin_path, filename)
            output_image_path = os.path.join(seg_path, filename)
            
            # 调用 Cell 1 中的算法
            processed_image = process_image(input_image_path)

            if processed_image is not None:
                # 确保保存的图片与原名一致
                cv2.imwrite(output_image_path, processed_image)
            else:
                print(f"\n⚠️ 警告：文件 {filename} 处理返回 None，已跳过。")
                
    except Exception as e:
        print(f"\n❌ 代码运行崩溃于文件: {filename}")
        print(f"❌ 错误详情: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        end_time = time.time()
        print("\n=== 模拟批量评测结束 ===")
        print(f"💾 结果已输出至：{os.path.abspath(seg_path)}")
        print(f"⏱️ 总耗时: {end_time - start_time:.2f} 秒")
else:
    print("⏹️ 没有文件被处理。")

=== 开始模拟批量评测 ===
🔍 在 '.\Origin' 中找到 2 个待处理文件。


图像处理进度: 100%|██████████| 2/2 [00:00<00:00, 14.03it/s]


=== 模拟批量评测结束 ===
💾 结果已输出至：c:\Users\Lenovo\Desktop\ProjDIP\SubmitExample\Seg
⏱️ 总耗时: 0.15 秒
